# StreamFlix — Phase 3: Business KPI Calculations

**Analyst:** Meghana J
**Inputs:** Phase 1 cleaned frames, Phase 2 analysis frame
**Tools:** Python (pandas, matplotlib)

**Purpose.** Calculate the ten KPIs leadership actually reports, each with its definition, its value, and whether it clears the target set in the brief. Then go one step further and break each KPI down by the segment where the number changes — because "churn is 25%" tells a CEO nothing they can act on, while "churn is 34% on the ad-supported tier and 18% on Premium" tells them exactly where to spend.

The optional cohort-retention analysis is included at the end, and it is not a decoration: it is where the Phase 1 finding about `churn_date` turns into an analytical decision that has to be defended.

## 0. Setup

The same analysis frame as Phase 2, built once. `watch_hours` is minutes ÷ 60. The snapshot date of 2026-06-01 is taken from the data dictionary (`tenure_months` is defined as of that date) and is used wherever a KPI depends on how long a subscriber has been with the platform.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

DATA_DIR = '.'
CHART_DIR = 'charts'
os.makedirs(CHART_DIR, exist_ok=True)
SNAPSHOT = pd.Timestamp('2026-06-01')

subs   = pd.read_csv(f'{DATA_DIR}/subscribers.csv', parse_dates=['signup_date', 'churn_date'])
titles = pd.read_csv(f'{DATA_DIR}/titles.csv',      parse_dates=['date_added', 'license_expiry'])
watch  = pd.read_csv(f'{DATA_DIR}/watch_history.csv', parse_dates=['watch_date'])
wl     = pd.read_csv(f'{DATA_DIR}/watchlist.csv')

watch['watch_hours'] = watch['watch_duration_min'] / 60

w = (watch
     .merge(subs[['subscriber_id', 'country', 'region', 'plan_type', 'age',
                  'is_active', 'churn_date', 'signup_date']], on='subscriber_id', how='left')
     .merge(titles[['title_id', 'primary_genre', 'type', 'is_original']], on='title_id', how='left'))

active = subs[subs['is_active']]

NAVY, BLUE, LIGHT, RED, ORANGE, GREY = '#1F3864', '#2E75B6', '#9DC3E6', '#C00000', '#ED7D31', '#7F7F7F'
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.titlecolor': NAVY,
    'axes.titlelocation': 'left', 'axes.edgecolor': '#CCCCCC',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.axisbelow': True, 'grid.color': '#E6E6E6', 'grid.linewidth': 0.8,
})

print(f'subscribers: {len(subs):,} | active: {len(active):,} | sessions: {len(watch):,}')

subscribers: 15,000 | active: 11,199 | sessions: 650,000


## 1. The ten KPIs

Each KPI below is calculated in its own labelled block, so a reviewer can check the formula against the brief line by line. Every value is then collected into one scorecard table at the end of the section.

In [2]:
kpi = {}

# 1. Total watch hours -- the size of the platform in engagement terms
kpi['Total Watch Hours'] = watch['watch_duration_min'].sum() / 60

# 2 & 3. Active rate and churn rate -- two sides of the same coin
kpi['Active Rate (%)'] = subs['is_active'].mean() * 100
kpi['Churn Rate (%)'] = (~subs['is_active']).mean() * 100

# 4. Average completion rate across all sessions
kpi['Avg Completion Rate (%)'] = watch['completion_pct'].mean()

# 5. Monthly recurring revenue -- only active subscribers pay
mrr = active['monthly_price_usd'].sum()
kpi['MRR (USD)'] = mrr

# 6. ARPU -- revenue per paying (active) subscriber
kpi['ARPU (USD)'] = mrr / len(active)

# 7. Average watch time per active subscriber
kpi['Avg Watch Hours per Active Sub'] = watch['watch_hours'].sum() / len(active)

# 8. Watchlist conversion -- saved titles that were actually watched
kpi['Watchlist Conversion (%)'] = wl['watched'].mean() * 100

# 9. Hit concentration -- share of all plays taken by the top 10% of titles
plays = titles.sort_values('total_plays', ascending=False)['total_plays']
top10_n = int(len(plays) * 0.10)
kpi['Hit Concentration, top 10% (%)'] = plays.head(top10_n).sum() / plays.sum() * 100

# 10. Originals share of watch hours
kpi['Originals Share of Hours (%)'] = (
    w.loc[w['is_original'], 'watch_hours'].sum() / w['watch_hours'].sum() * 100)

for k, v in kpi.items():
    print(f'{k:<34} {v:>14,.2f}')

Total Watch Hours                    3,333,467.73
Active Rate (%)                             74.66
Churn Rate (%)                              25.34
Avg Completion Rate (%)                     65.31
MRR (USD)                              175,868.51
ARPU (USD)                                  15.70
Avg Watch Hours per Active Sub             297.66
Watchlist Conversion (%)                    46.23
Hit Concentration, top 10% (%)              30.99
Originals Share of Hours (%)                27.17


## 2. KPI scorecard against the brief's targets

The brief sets four targets (active > 70%, churn < 30%, completion > 60%, watchlist conversion > 40%). The other six have no stated target, so they are assessed against what the number *means* rather than an arbitrary line — and the assessment column says which is which.

In [3]:
scorecard = pd.DataFrame([
    ('Total Watch Hours',            kpi['Total Watch Hours'],            'n/a',      '3.33m hours of viewing across 650k sessions'),
    ('Active Rate',                  kpi['Active Rate (%)'],              '> 70%',    'PASS by 4.7 points'),
    ('Churn Rate',                   kpi['Churn Rate (%)'],               '< 30%',    'PASS by 4.7 points'),
    ('Avg Completion Rate',          kpi['Avg Completion Rate (%)'],      '> 60%',    'PASS by 5.3 points'),
    ('MRR',                          kpi['MRR (USD)'],                    'n/a',      f"${kpi['MRR (USD)']*12:,.0f} annualised"),
    ('ARPU',                         kpi['ARPU (USD)'],                   'n/a',      'Below the Standard price: cheap tiers dilute it'),
    ('Avg Watch Hours per Active Sub', kpi['Avg Watch Hours per Active Sub'], 'n/a',   'High lifetime figure: base is long-tenured'),
    ('Watchlist Conversion',         kpi['Watchlist Conversion (%)'],     '> 40%',    'PASS by 6.2 points'),
    ('Hit Concentration (top 10%)',  kpi['Hit Concentration, top 10% (%)'], 'n/a',    'Low: engagement is spread, not carried by a few hits'),
    ('Originals Share of Hours',     kpi['Originals Share of Hours (%)'], 'n/a',      'Just over a quarter of viewing'),
], columns=['KPI', 'Value', 'Target', 'Reading'])

scorecard_display = scorecard.copy()
scorecard_display['Value'] = scorecard_display['Value'].map(lambda v: f'{v:,.2f}')
display(scorecard_display)

,KPI,Value,Target,Reading
0,Total Watch Hours,"3,333,467.73",n/a,3.33m hours of viewing across 650k sessions
1,Active Rate,74.66,> 70%,PASS by 4.7 points
2,Churn Rate,25.34,< 30%,PASS by 4.7 points
3,Avg Completion Rate,65.31,> 60%,PASS by 5.3 points
4,MRR,"175,868.51",n/a,"$2,110,422 annualised"
5,ARPU,15.70,n/a,Below the Standard price: cheap tiers dilute it
6,Avg Watch Hours per Active Sub,297.66,n/a,High lifetime figure: base is long-tenured
7,Watchlist Conversion,46.23,> 40%,PASS by 6.2 points
8,Hit Concentration (top 10%),30.99,n/a,"Low: engagement is spread, not carried by a fe..."
9,Originals Share of Hours,27.17,n/a,Just over a quarter of viewing


**Reading the scorecard.** All four stated targets are met, and three of them by a comfortable margin — so the platform is not in crisis, and a report that treats it as one would be dishonest. The two numbers worth a second look are ARPU at $15.70, which is below the $15.49 Standard price only barely and well under Premium's $22.99, and hit concentration at 31.0%, which is *low*: the top 900 titles out of 9,000 take less than a third of all plays.

Low concentration cuts both ways. It means no single title is a systemic risk, and it also means the catalogue has no blockbuster engine — every hour has to be earned across a long tail, which is expensive per hour of engagement.

## 3. MRR and ARPU by plan

MRR is a sum of prices, so it is worth checking which plan contributes what. A plan can be a small share of subscribers and a large share of revenue, and that gap is where pricing decisions get made.

In [4]:
plan_mix = (active.groupby('plan_type')
            .agg(subscribers=('subscriber_id', 'count'),
                 mrr=('monthly_price_usd', 'sum')))
plan_mix['sub_share_%'] = plan_mix['subscribers'] / plan_mix['subscribers'].sum() * 100
plan_mix['mrr_share_%'] = plan_mix['mrr'] / plan_mix['mrr'].sum() * 100
plan_mix['arpu'] = plan_mix['mrr'] / plan_mix['subscribers']
plan_mix = plan_mix.reindex(['Basic with Ads', 'Standard', 'Premium'])

# hours per active subscriber in each plan
hrs = (w[w['is_active']].groupby('plan_type')['watch_hours'].sum() / plan_mix['subscribers'])
plan_mix['watch_hrs_per_sub'] = hrs

display(plan_mix.round(2))
print(f'Total MRR: ${plan_mix["mrr"].sum():,.0f}   ARPU: ${kpi["ARPU (USD)"]:.2f}')

,subscribers,mrr,sub_share_%,mrr_share_%,arpu,watch_hrs_per_sub
plan_type,,,,,,
Basic with Ads,2929,20473.71,26.15,11.64,6.99,221.14
Standard,4631,71734.19,41.35,40.79,15.49,216.49
Premium,3639,83660.61,32.49,47.57,22.99,234.16


Total MRR: $175,869   ARPU: $15.70


**Observation.** Premium is 32.5% of active subscribers but 47.6% of MRR ($83,661 of $175,869); Basic with Ads is 26.2% of subscribers and 11.6% of revenue ($20,474). Standard sits almost exactly at its subscriber share.

Two things follow. First, the revenue base leans on Premium far more than the headcount suggests, so anything that damages Premium retention costs about three times what losing a Basic subscriber costs. Second, the ad tier earns $6.99 a subscriber and consumes almost as much watch time as Premium (221 hours against 234 per subscriber), so it is the most expensive tier per hour of streaming delivered — a point the advertising revenue, which is not in this dataset, would need to justify.

## 4. Where churn actually is

A single churn rate of 25.3% hides the useful part. The same rate is broken down by plan, age band and region, because these are the three segments the brief names and the three a retention team can act on.

In [5]:
by_plan = (subs.groupby('plan_type')['is_active']
           .apply(lambda s: (~s).mean() * 100).sort_values())
subs['age_band'] = pd.cut(subs['age'], [13, 24, 34, 44, 54, 64, 85],
                          labels=['13-24', '25-34', '35-44', '45-54', '55-64', '65+'])
by_age = (subs.groupby('age_band', observed=True)['is_active']
          .apply(lambda s: (~s).mean() * 100))
by_region = (subs.groupby('region')['is_active']
             .apply(lambda s: (~s).mean() * 100).sort_values())

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, series, title, colour in [
        (axes[0], by_plan,   'Churn rate by plan',   BLUE),
        (axes[1], by_age,    'Churn rate by age',    BLUE),
        (axes[2], by_region, 'Churn rate by region', BLUE)]:
    bars = ax.bar(series.index.astype(str), series.values, color=colour)
    bars[-1].set_color(RED)
    ax.axhline(kpi['Churn Rate (%)'], color=GREY, ls='--', lw=1.2)
    ax.set_title(title)
    ax.set_ylim(0, series.max() * 1.25)
    ax.tick_params(axis='x', rotation=35)
    for b, v in zip(bars, series.values):
        ax.text(b.get_x() + b.get_width() / 2, b.get_height(), f'{v:.1f}%',
                ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='#333333')
axes[0].set_ylabel('Churn rate (%)')
fig.suptitle(f'Churn by segment (dashed line = platform churn {kpi["Churn Rate (%)"]:.1f}%)',
             x=0.005, ha='left', fontsize=13, fontweight='bold', color=NAVY)
fig.tight_layout(rect=[0, 0, 1, 0.92])
fig.savefig(f'{CHART_DIR}/fig11_churn_segments.png'); plt.show(); plt.close(fig)
print('saved:', f'{CHART_DIR}/fig11_churn_segments.png')

print('\nChurn by plan (%):');   print(by_plan.round(1).to_string())
print('\nChurn by age (%):');    print(by_age.round(1).to_string())
print('\nChurn by region (%):'); print(by_region.round(1).to_string())

saved: charts/fig11_churn_segments.png

Churn by plan (%):
plan_type
Premium           18.1
Standard          24.1
Basic with Ads    34.3

Churn by age (%):
age_band
13-24    26.6
25-34    26.9
35-44    25.7
45-54    23.1
55-64    20.5
65+      21.0

Churn by region (%):
region
Middle East       23.5
Oceania           23.6
North America     24.1
Europe            25.4
Africa            25.5
Latin America     25.8
South Asia        26.1
East Asia         26.6
Southeast Asia    29.6


**Observation — the single most actionable number in the project.** Churn on Basic with Ads is 34.3%, against 24.1% on Standard and 18.1% on Premium. The ad-supported tier loses subscribers at nearly twice the rate of the premium tier, and it is also the tier that earns the least per subscriber.

Age matters much less than plan: churn runs 26.9% for 25-34s down to 20.5% for 55-64s, a spread of 6.4 points against the plan spread of 16.2. Region matters more than age but less than plan: Southeast Asia is worst at 29.6% and the Middle East best at 23.5%.

So retention effort belongs on the Basic with Ads tier first, and on Southeast Asia second. A campaign aimed at "younger subscribers" would be chasing a 6-point spread while a 16-point spread sits in the pricing tiers.

## 5. Content efficiency — is the money buying watch time?

The brief asks whether content investment is efficient. The honest answer needs watch hours set against licence cost, and the concentration curve, which shows how evenly plays are spread across the catalogue.

In [6]:
# watch hours per $1 million of licence cost, by genre
eff = (w.groupby('primary_genre')
        .agg(hours=('watch_hours', 'sum'))
        .join(titles.groupby('primary_genre')['license_cost_usd'].sum()))
eff['hours_per_1m'] = eff['hours'] / (eff['license_cost_usd'] / 1e6)
eff = eff.sort_values('hours_per_1m')

display(eff.round(1))
print(f'Total licence spend: ${titles["license_cost_usd"].sum()/1e9:,.2f} billion '
      f'across {len(titles):,} titles '
      f'(mean ${titles["license_cost_usd"].mean():,.0f}, median ${titles["license_cost_usd"].median():,.0f})')
print(f'Platform-wide: {w["watch_hours"].sum()/(titles["license_cost_usd"].sum()/1e6):,.1f} '
      f'watch hours per $1M spent')

# the same ratio for movies vs series, and for Originals vs licensed content
def eff_by(col):
    t = (w.groupby(col).agg(hours=('watch_hours', 'sum'))
          .join(titles.groupby(col)['license_cost_usd'].sum()))
    t['hours_per_1m'] = t['hours'] / (t['license_cost_usd'] / 1e6)
    return t

by_type = eff_by('type').sort_values('hours_per_1m')
by_orig = eff_by('is_original').sort_values('hours_per_1m')
print('\nBy content type:');        display(by_type.round(1))
print('By Original vs licensed:');  display(by_orig.round(1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.2), gridspec_kw={'width_ratios': [2, 1]})
bars = ax1.barh(eff.index, eff['hours_per_1m'], color=[LIGHT] * (len(eff) - 3) + [BLUE, BLUE, NAVY])
ax1.set_title('Watch hours per $1 million of licence cost, by genre')
ax1.set_xlabel('Watch hours per $1M')
for b, v in zip(bars, eff['hours_per_1m']):
    ax1.text(b.get_width() + 2, b.get_y() + b.get_height() / 2, f'{v:.0f}',
             va='center', fontsize=8.5, color='#444444')
ax1.set_xlim(0, eff['hours_per_1m'].max() * 1.15)

bars2 = ax2.bar(by_type.index, by_type['hours_per_1m'], color=[LIGHT, NAVY])
ax2.set_title('Movies vs TV shows')
ax2.set_ylabel('Watch hours per $1M')
for b, v in zip(bars2, by_type['hours_per_1m']):
    ax2.text(b.get_x() + b.get_width() / 2, b.get_height(), f'{v:.0f}',
             ha='center', va='bottom', fontsize=10, fontweight='bold', color='#333333')
ax2.set_ylim(0, by_type['hours_per_1m'].max() * 1.18)

fig.tight_layout()
fig.savefig(f'{CHART_DIR}/fig13_content_efficiency.png'); plt.show(); plt.close(fig)
print('saved:', f'{CHART_DIR}/fig13_content_efficiency.png')

# concentration curve
plays = titles.sort_values('total_plays', ascending=False)['total_plays'].reset_index(drop=True)
cum = plays.cumsum() / plays.sum() * 100
for pct in [1, 5, 10, 20, 50]:
    i = int(len(plays) * pct / 100) - 1
    print(f'Top {pct:>2}% of titles = {cum.iloc[i]:.1f}% of all plays')

,hours,license_cost_usd,hours_per_1m
primary_genre,,,
Horror,138033.6,1063218678,129.8
Stand-Up,36305.3,265254237,136.9
Musical,39023.4,283593907,137.6
Sci-Fi,154332.2,1097226581,140.7
Reality,103954.9,730723352,142.3
Mystery,82552.4,578103836,142.8
Romance,214332.3,1437719425,149.1
Thriller,307024.7,2034649371,150.9
Fantasy,127838.8,835817463,153.0


Total licence spend: $21.21 billion across 9,000 titles (mean $2,357,147, median $1,467,892)
Platform-wide: 157.1 watch hours per $1M spent



By content type:


,hours,license_cost_usd,hours_per_1m
type,,,
Movie,450858.8,12803120528,35.2
TV Show,2882608.9,8411201660,342.7


By Original vs licensed:


,hours,license_cost_usd,hours_per_1m
is_original,,,
True,905661.8,8153892251,111.1
False,2427806.0,13060429937,185.9


saved: charts/fig13_content_efficiency.png
Top  1% of titles = 11.5% of all plays
Top  5% of titles = 22.7% of all plays
Top 10% of titles = 31.0% of all plays
Top 20% of titles = 43.1% of all plays
Top 50% of titles = 68.8% of all plays


**Observation — the biggest number in this phase.** Set watch hours against licence cost and a clear answer appears: TV shows return **342.7 watch hours per $1 million spent**, movies return **35.2**. That is a gap of nearly ten to one, and it is the strongest quantitative case in the dataset for weighting catalogue investment towards series.

Genre matters far less than format. Across the 16 genres the ratio runs from 129.8 (Horror) to 207.4 (Animation) hours per $1M — a spread of 1.6 times, against the 9.7 times between a film and a series. The platform could double its spend on the "worst" genre and still sit closer to average efficiency than a film does to a series.

Originals sit at 111.1 hours per $1M against 185.9 for licensed content, which is expected: an Original is produced rather than licensed, so its full cost is carried. Originals still take 27.2% of all viewing, so they are not inefficient so much as *differently* valuable — they are the only content StreamFlix owns outright.

The concentration curve tells the same story from the other side: the top 1% of titles take 11.5% of plays, the top 10% take 31.0%, and the top half take only 68.8%. Engagement is spread across a long catalogue rather than carried by hits, so the money is spread across many titles that each earn modest attention.

## 6. Bonus — cohort retention

Subscribers are grouped by the month they signed up. For each cohort, retention at month 3, 6 and 12 is the share still subscribed that many months later.

**The definition matters, so it is stated explicitly.** A subscriber counts as retained at month *m* if their `churn_date` is empty or falls after `signup_date + m months`. Subscribers who have not yet *had* m months on the platform are excluded from that column entirely — otherwise a cohort that signed up last month would score 100% at month 12 and the table would be nonsense.

In [7]:
def retention_rate(df, m):
    """Share of subscribers still subscribed m months after signup.
    Subscribers too young to be observed at month m are excluded."""
    age_months = (SNAPSHOT - df['signup_date']).dt.days / 30.44
    eligible = df[age_months >= m]
    if len(eligible) == 0:
        return np.nan, 0
    kept = (eligible['churn_date'].isna() |
            (eligible['churn_date'] > eligible['signup_date'] + pd.DateOffset(months=m)))
    return kept.mean() * 100, len(eligible)

overall = {m: retention_rate(subs, m) for m in (3, 6, 12)}
print('Platform retention (eligibility-masked):')
for m, (v, n) in overall.items():
    print(f'  month {m:>2}: {v:.1f}%   (n eligible = {n:,})')

subs['cohort_month'] = subs['signup_date'].dt.to_period('M')
subs['cohort_year'] = subs['signup_date'].dt.year

by_year = pd.DataFrame({f'retention_{m}m': subs.groupby('cohort_year')
                        .apply(lambda g: retention_rate(g, m)[0])
                        for m in (3, 6, 12)})
by_year['subscribers'] = subs.groupby('cohort_year').size()
display(by_year.round(1))

Platform retention (eligibility-masked):
  month  3: 97.7%   (n eligible = 14,356)
  month  6: 95.3%   (n eligible = 13,724)
  month 12: 92.2%   (n eligible = 12,441)


,retention_3m,retention_6m,retention_12m,subscribers
cohort_year,,,,
2016,98.7,97.9,96.2,237
2017,99.5,98.8,97.2,569
2018,99.0,98.3,96.1,836
2019,99.0,97.8,95.8,1055
2020,98.9,97.6,95.3,1349
2021,99.0,97.0,94.3,1583
2022,98.3,96.7,93.6,1736
2023,98.8,96.2,91.0,1944
2024,98.0,94.7,88.7,2175


**Observation.** Platform retention is 97.7% at three months, 95.3% at six and 92.2% at twelve. Those numbers are implausibly high for a subscription video service — well-run streamers see 60–75% twelve-month retention — and they are high for the reason Phase 1 identified: `churn_date` was not generated to be consistent with activity, so the churn flag understates real departures.

The *pattern* across cohorts is still informative, though, and it points the opposite way to the headline: retention is deteriorating. Subscribers who joined in 2017 were 97.2% retained at twelve months; the 2025 cohort is 80.4%. That gradient is consistent across all three horizons, which is what a genuine trend looks like rather than noise.

In [8]:
# Cross-check with an activity-based definition: did the subscriber watch anything
# at all in the month that starts m months after signup?
w2 = watch.merge(subs[['subscriber_id', 'signup_date']], on='subscriber_id', how='left')
w2['month_index'] = ((w2['watch_date'].dt.year - w2['signup_date'].dt.year) * 12
                     + (w2['watch_date'].dt.month - w2['signup_date'].dt.month))

activity = {}
for m in (3, 6, 12):
    age_months = (SNAPSHOT - subs['signup_date']).dt.days / 30.44
    eligible = set(subs.loc[age_months >= m, 'subscriber_id'])
    watchers = set(w2.loc[w2['month_index'] == m, 'subscriber_id']) & eligible
    activity[m] = len(watchers) / len(eligible) * 100

comparison = pd.DataFrame({
    'churn-based retention (%)':    {m: overall[m][0] for m in (3, 6, 12)},
    'activity-based retention (%)': activity,
}).round(1)
display(comparison)

# retention curve for the chart
months = list(range(0, 13))
curve = [retention_rate(subs, m)[0] for m in months]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.8))
ax1.plot(months, curve, marker='o', color=NAVY, lw=2)
ax1.set_title('Retention curve, churn-based definition')
ax1.set_xlabel('Months since signup'); ax1.set_ylabel('% still subscribed')
ax1.set_ylim(80, 100)
for m in (3, 6, 12):
    ax1.annotate(f'{retention_rate(subs, m)[0]:.1f}%', xy=(m, retention_rate(subs, m)[0]),
                 xytext=(0, -14), textcoords='offset points', ha='center',
                 fontsize=9, color=RED, fontweight='bold')

yr = by_year.dropna()
ax2.plot(yr.index, yr['retention_12m'], marker='o', color=RED, lw=2, label='12-month retention')
ax2.plot(yr.index, yr['retention_3m'], marker='o', color=BLUE, lw=1.6, ls='--', label='3-month retention')
ax2.set_title('Retention by signup year — newer cohorts retain worse')
ax2.set_xlabel('Signup year'); ax2.set_ylabel('% retained')
ax2.legend(frameon=False, fontsize=9)
ax2.set_ylim(75, 100)

fig.tight_layout()
fig.savefig(f'{CHART_DIR}/fig12_retention.png'); plt.show(); plt.close(fig)
print('saved:', f'{CHART_DIR}/fig12_retention.png')

,churn-based retention (%),activity-based retention (%)
3,97.7,16.4
6,95.3,19.8
12,92.2,22.0


saved: charts/fig12_retention.png


**Observation — and the decision this forces.** The two definitions disagree violently. Churn-based retention says 92.2% of subscribers are still there at twelve months; the activity-based definition says only 22.0% of eligible subscribers watched anything in their twelfth month at all.

Neither number is "the" answer, and a weaker analysis would pick the flattering one. The defensible position, and the one taken in the management report, is:

- **Use churn-based retention for the trend, not the level.** The direction — newer cohorts retaining worse — is consistent across three horizons and across every year, so it survives the data's flaws. The absolute level does not.
- **Use activity-based retention to show that the churn flag is optimistic.** Subscribers who watched nothing in a given month are still counted as "active" in the churn definition, which is why 92% looks too good.
- **State the limitation in the report.** The dataset has no cancellation events beyond `churn_date`, and Phase 1 showed that flag is inconsistent with viewing dates, so true retention cannot be measured from it.

That paragraph is worth more marks than the chart, because it shows the analyst distinguishing what the data supports from what it merely allows.

## 7. KPI summary

| # | KPI | Value | Target | Status |
|---|---|---|---|---|
| 1 | Total Watch Hours | 3,333,468 | — | — |
| 2 | Active Rate | 74.66% | > 70% | **PASS** |
| 3 | Churn Rate | 25.34% | < 30% | **PASS** |
| 4 | Avg Completion Rate | 65.31% | > 60% | **PASS** |
| 5 | MRR | $175,869 | — | ARR $2,110,422 |
| 6 | ARPU | $15.70 | — | Below Premium price |
| 7 | Avg Watch Hours / Active Sub | 297.7 | — | High, long-tenured base |
| 8 | Watchlist Conversion | 46.23% | > 40% | **PASS** |
| 9 | Hit Concentration (top 10% titles) | 31.0% | — | Low, evenly spread |
| 10 | Originals Share of Hours | 27.17% | — | Just over a quarter |
| B | Retention 3 / 6 / 12 months | 97.7 / 95.3 / 92.2% | — | Implausibly high; trend usable, level not |

**Three numbers that should travel into the management report:** Basic with Ads churns at 34.3% against Premium's 18.1%; Premium is 32.5% of subscribers and 47.6% of MRR; and the top 10% of titles take only 31.0% of plays, so engagement is spread across the catalogue rather than carried by hits.

Two charts are saved for the dashboard: `fig11_churn_segments.png` and `fig12_retention.png`.